Online ML notebook: Feature Store and real-time inference pipeline
*Co-authored with CoCo*

# Getting Started with Online ML - Feature Store and Inference

This notebook walks through building a real-time product recommendation service.

**Prerequisites:** Run `setup.sql` before starting.

**Packages required:** Add `snowflake-ml-python>=1.41.0`, `xgboost`, `scikit-learn` via the Packages picker.

## Connect and Import Libraries

In [ ]:
# Cell 1 - Connect and Import Libraries
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

session = get_active_session()
session.use_database("RECOMMEND_DB")
session.use_schema("RECOMMEND")
session.use_warehouse("RECOMMEND_WH")
print(f"Connected as: {session.get_current_user()}")

## Data Preparation

Generate synthetic e-commerce data: 1,000 users, 500 items, 50,000 interactions.

In [ ]:
# Cell 2 - Generate synthetic data
np.random.seed(42)
N_USERS, N_ITEMS, N_INTERACTIONS = 1000, 500, 50_000
categories = ["Electronics", "Clothing", "Books", "Sports", "Home"]
price_ranges = ["Low", "Medium", "High"]

users_df = pd.DataFrame({
    "USER_ID": [f"user_{i:04d}" for i in range(N_USERS)],
    "AGE_GROUP": np.random.choice(["18-24","25-34","35-44","45-54","55+"], N_USERS),
    "PREFERRED_CATEGORY": np.random.choice(categories, N_USERS),
})
items_df = pd.DataFrame({
    "ITEM_ID": [f"item_{i:04d}" for i in range(N_ITEMS)],
    "CATEGORY": np.random.choice(categories, N_ITEMS),
    "PRICE_RANGE": np.random.choice(price_ranges, N_ITEMS),
    "AVG_RATING": np.round(np.random.uniform(2.5, 5.0, N_ITEMS), 1),
})

def generate_interactions(users_df, items_df, n):
    user_ids = np.random.choice(users_df["USER_ID"], n)
    item_ids = np.random.choice(items_df["ITEM_ID"], n)
    user_cat = users_df.set_index("USER_ID")["PREFERRED_CATEGORY"]
    item_cat = items_df.set_index("ITEM_ID")["CATEGORY"]
    cat_match = (user_cat[user_ids].values == item_cat[item_ids].values)
    click_prob = np.where(cat_match, 0.35, 0.10)
    clicked = np.random.binomial(1, click_prob)
    purchased = clicked * np.random.binomial(1, 0.15, n)
    ts = [datetime.now() - timedelta(days=np.random.randint(0, 60)) for _ in range(n)]
    return pd.DataFrame({"USER_ID": user_ids, "ITEM_ID": item_ids,
                          "CLICKED": clicked, "PURCHASED": purchased, "TS": ts})

interactions_df = generate_interactions(users_df, items_df, N_INTERACTIONS)
session.write_pandas(users_df, "USERS", overwrite=True, auto_create_table=True)
session.write_pandas(items_df, "ITEMS", overwrite=True, auto_create_table=True)
session.write_pandas(interactions_df, "INTERACTIONS", overwrite=True, auto_create_table=True)
print(f"Users: {len(users_df):,} | Items: {len(items_df):,} | Interactions: {len(interactions_df):,}")

In [ ]:
# Cell 3 - Create feature engineering views (pre-encoded for model input)

# User features: activity metrics + category encoded as integer
session.sql("""
CREATE OR REPLACE VIEW RECOMMEND_DB.RECOMMEND.USER_FEATURES_V AS
SELECT
    u.USER_ID,
    COALESCE(SUM(CASE WHEN i.PURCHASED = 1
                       AND TO_TIMESTAMP(i.TS) >= DATEADD(day, -30, CURRENT_TIMESTAMP())
                  THEN 1 ELSE 0 END), 0)    AS PURCHASE_COUNT_30D,
    COALESCE(AVG(CASE WHEN TO_TIMESTAMP(i.TS) >= DATEADD(day, -7, CURRENT_TIMESTAMP())
                 THEN i.CLICKED END), 0)     AS CLICK_RATE_7D,
    CASE u.PREFERRED_CATEGORY
        WHEN 'Books' THEN 0 WHEN 'Clothing' THEN 1 WHEN 'Electronics' THEN 2
        WHEN 'Home' THEN 3 WHEN 'Sports' THEN 4 ELSE 0
    END AS PREFERRED_CATEGORY_ENC,
    COALESCE(MAX(TO_TIMESTAMP(i.TS)), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ AS LAST_ACTIVITY_TS
FROM
    RECOMMEND_DB.RECOMMEND.USERS u
LEFT JOIN
    RECOMMEND_DB.RECOMMEND.INTERACTIONS i ON u.USER_ID = i.USER_ID
GROUP BY
    u.USER_ID, u.PREFERRED_CATEGORY
""").collect()
print("USER_FEATURES_V created.")

# Item features: category and price encoded as integers + rating
session.sql("""
CREATE OR REPLACE VIEW RECOMMEND_DB.RECOMMEND.ITEM_FEATURES_V AS
SELECT
    ITEM_ID,
    CASE CATEGORY
        WHEN 'Books' THEN 0 WHEN 'Clothing' THEN 1 WHEN 'Electronics' THEN 2
        WHEN 'Home' THEN 3 WHEN 'Sports' THEN 4 ELSE 0
    END AS ITEM_CATEGORY_ENC,
    CASE PRICE_RANGE
        WHEN 'High' THEN 0 WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2 ELSE 0
    END AS PRICE_RANGE_ENC,
    AVG_RATING
FROM
    RECOMMEND_DB.RECOMMEND.ITEMS
""").collect()
print("ITEM_FEATURES_V created.")

## Feature Store Setup

Initialize the Feature Store, provision the Postgres online service, and register the Feature View.

> **Note:** `create_online_service()` takes a few minutes on first run.

In [ ]:
# NOTE: create_online_service() requires a PAID Snowflake account.
# It provisions HIGHMEM compute internally, which is NOT available on trial accounts.
# If you are on a trial account, this cell will fail with a HIGHMEM_2XL error.

# Cell 4 - Initialize Feature Store and create online service
import time
from snowflake.ml.feature_store import FeatureStore

fs = FeatureStore(
    session=session,
    database="RECOMMEND_DB",
    name="RECOMMEND",
    default_warehouse="RECOMMEND_WH",
    creation_mode="CREATE_IF_NOT_EXIST",
)

# Create online service (skip if already exists)
try:
    print("Creating online service (takes a few minutes)...")
    fs.create_online_service("FS_PRODUCER_ROLE", "FS_CONSUMER_ROLE")
except Exception as e:
    if "already exists" in str(e):
        print("Online service already exists, skipping creation.")
    else:
        raise

status = fs.get_online_service_status()
while status.status != "RUNNING":
    time.sleep(30)
    status = fs.get_online_service_status()
    print(f"Status: {status.status}")

print(f"Online service RUNNING. Endpoints: {status.endpoints}")

In [ ]:
# Cell 5 - Register entities and Feature Views with online store
from snowflake.ml.feature_store import FeatureView, Entity, OnlineConfig, OnlineStoreType

# User entity and feature view
user_entity = Entity(name="USER", join_keys=["USER_ID"])
fs.register_entity(user_entity)

# Delete old feature views if they exist (schema may have changed)
try:
    fs.delete_feature_view(fs.get_feature_view("USER_FEATURES", "V1"))
except:
    pass
try:
    fs.delete_feature_view(fs.get_feature_view("ITEM_FEATURES", "V1"))
except:
    pass

user_fv = FeatureView(
    name="USER_FEATURES",
    entities=[user_entity],
    feature_df=session.table("RECOMMEND_DB.RECOMMEND.USER_FEATURES_V"),
    timestamp_col="LAST_ACTIVITY_TS",
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10 seconds",
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc="User activity features for product recommendation",
)
user_fv_registered = fs.register_feature_view(user_fv, version="V1", block=True)
print(f"Registered: {user_fv_registered.name}/{user_fv_registered.version}")

# Item entity and feature view
item_entity = Entity(name="ITEM", join_keys=["ITEM_ID"])
fs.register_entity(item_entity)

item_fv = FeatureView(
    name="ITEM_FEATURES",
    entities=[item_entity],
    feature_df=session.table("RECOMMEND_DB.RECOMMEND.ITEM_FEATURES_V"),
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10 seconds",
        store_type=OnlineStoreType.POSTGRES,
    ),
    desc="Item features for product recommendation",
)
item_fv_registered = fs.register_feature_view(item_fv, version="V1", block=True)
print(f"Registered: {item_fv_registered.name}/{item_fv_registered.version}")

In [ ]:
# Cell 6 - Verify feature retrieval from both Feature Views
sample_users = pd.DataFrame({"USER_ID": ["user_0001", "user_0042", "user_0107"]})
user_feats = fs.retrieve_feature_values(
    spine_df=session.create_dataframe(sample_users),
    features=[user_fv_registered],
)
print("=== User Features ===")
user_feats.show()

sample_items = pd.DataFrame({"ITEM_ID": ["item_0010", "item_0055", "item_0200"]})
item_feats = fs.retrieve_feature_values(
    spine_df=session.create_dataframe(sample_items),
    features=[item_fv_registered],
)
print("\n=== Item Features ===")
item_feats.show()

## Model Building and Deployment

In [ ]:
# Cell 7 - Prepare training data (features already encoded in Feature Views)
all_users = session.table("USERS").select("USER_ID").to_pandas()
user_features = fs.retrieve_feature_values(
    spine_df=session.create_dataframe(all_users),
    features=[user_fv_registered],
).to_pandas()

all_items = session.table("ITEMS").select("ITEM_ID").to_pandas()
item_features = fs.retrieve_feature_values(
    spine_df=session.create_dataframe(all_items),
    features=[item_fv_registered],
).to_pandas()

interactions = session.table("INTERACTIONS").to_pandas()

train_df = (
    interactions
    .merge(user_features, on="USER_ID")
    .merge(item_features, on="ITEM_ID")
)

FEATURE_COLS = ["PURCHASE_COUNT_30D", "CLICK_RATE_7D", "PREFERRED_CATEGORY_ENC",
                "ITEM_CATEGORY_ENC", "PRICE_RANGE_ENC", "AVG_RATING"]
X = train_df[FEATURE_COLS]
y = train_df["CLICKED"]
print(f"Training samples: {len(X):,}  |  Click rate: {y.mean():.2%}")

In [ ]:
# Cell 8 - Train XGBoost model
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    eval_metric="auc", random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f"Test ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.4f}")

In [ ]:
# Cell 9 - Register model in Snowflake Model Registry
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="RECOMMEND_DB", schema_name="RECOMMEND")
sample_input = pd.DataFrame([{"PURCHASE_COUNT_30D": 2, "CLICK_RATE_7D": 0.2,
    "PREFERRED_CATEGORY_ENC": 0, "ITEM_CATEGORY_ENC": 1,
    "PRICE_RANGE_ENC": 1, "AVG_RATING": 4.2}])

# Drop existing service and model if present (for clean re-runs)
try:
    session.sql("DROP SERVICE IF EXISTS RECOMMEND_DB.RECOMMEND.RECOMMEND_INFERENCE_SERVICE").collect()
except:
    pass
try:
    reg.delete_model("PRODUCT_RECOMMEND_MODEL")
except:
    pass

mv = reg.log_model(
    model=model, model_name="PRODUCT_RECOMMEND_MODEL", version_name="V1",
    sample_input_data=sample_input,
    conda_dependencies=["xgboost", "scikit-learn", "pandas", "numpy"],
    comment="XGBoost click-prediction model with Feature Store lookup",
)
print(f"Registered: {mv.model_name}/{mv.version_name}")

In [ ]:
# Cell 10 - Deploy to SPCS with feature lookup from User Feature View
# Note: feature_sources_per_function currently supports max 1 lookup source.
# User features are auto-fetched; item features are passed in the request.
mv.create_service(
    service_name="RECOMMEND_INFERENCE_SERVICE",
    service_compute_pool="RECOMMEND_CPU_POOL",
    ingress_enabled=True,
    force_rebuild=True,
    feature_sources_per_function={
        "predict": [user_fv_registered],
    },
)
print("Service deploying... run Cell 11 to check status.")

In [ ]:
# Cell 11 - Check service status and get endpoint URL
services = mv.list_services()
print(services[["name", "status", "inference_endpoint", "internal_endpoint"]])

inference_ep = services["inference_endpoint"].iloc[0]
internal_ep = services["internal_endpoint"].iloc[0]

if inference_ep:
    print(f"\nPublic endpoint:\nhttps://{inference_ep}")
else:
    print(f"\nPublic endpoint not yet available (ingress provisioning).")
    print(f"Internal endpoint (usable within Snowflake):\n{internal_ep}")

## Real-Time Serving

Call the inference endpoint with only a `USER_ID`. The service fetches features from Postgres automatically.


In [ ]:
# Cell 12 - Inference with feature lookup
# Send USER_ID (auto-fetched from Feature Store) + item features in the request
import requests, json

services = mv.list_services()
inference_ep = services["inference_endpoint"].iloc[0]
internal_ep = services["internal_endpoint"].iloc[0]
ENDPOINT_URL = f"https://{inference_ep}/predict" if inference_ep else f"{internal_ep}/predict"

print(f"Calling: {ENDPOINT_URL}")
print("USER_ID triggers Feature Store lookup for user features.")
print("Item features (ITEM_CATEGORY_ENC, PRICE_RANGE_ENC, AVG_RATING) are sent directly.\n")

payload = {
    "dataframe_split": {
        "index": [0, 1, 2],
        "columns": ["USER_ID", "ITEM_CATEGORY_ENC", "PRICE_RANGE_ENC", "AVG_RATING"],
        "data": [
            ["user_0042", 0, 1, 4.5],  # Books, Low price
            ["user_0107", 3, 0, 3.8],  # Home, High price
            ["user_0255", 2, 2, 4.2],  # Electronics, Medium price
        ],
    }
}

token = session.connection._rest._token
headers = {"Authorization": f'Snowflake Token="{token}"', "Content-Type": "application/json"}

response = requests.post(ENDPOINT_URL, headers=headers, json=payload, timeout=60)
print(f"Status: {response.status_code}")
if response.ok:
    print(json.dumps(response.json(), indent=2))
else:
    print(f"Error: {response.text}")

In [ ]:
# Cell 13 - Override a user feature value (counterfactual / what-if analysis)
# Feature Store would normally provide PURCHASE_COUNT_30D, but we override it here.
payload_override = {
    "dataframe_split": {
        "index": [0],
        "columns": ["USER_ID", "PURCHASE_COUNT_30D", "ITEM_CATEGORY_ENC", "PRICE_RANGE_ENC", "AVG_RATING"],
        "data": [["user_0042", 20, 0, 1, 4.5]],
    }
}

response = requests.post(ENDPOINT_URL, headers=headers, json=payload_override, timeout=60)
print(f"Status: {response.status_code}")
if response.ok:
    print("Override result:", json.dumps(response.json(), indent=2))
    print("\nNote: PURCHASE_COUNT_30D=20 was sent directly (overriding Feature Store value).")
    print("Other user features (CLICK_RATE_7D, PREFERRED_CATEGORY_ENC) were fetched from Feature Store.")
else:
    print(f"Error: {response.text}")

## Cleanup
Run the cell below to stop all running services and release compute resources.
This prevents ongoing credit consumption after the demo.

In [ ]:
# Cell 14 - Cleanup: Stop services and release compute to avoid ongoing charges
print("=== Stopping credit-consuming resources ===")
print()

# 1. Drop the model inference service (SPCS)
try:
    session.sql("DROP SERVICE IF EXISTS RECOMMEND_DB.RECOMMEND.RECOMMEND_INFERENCE_SERVICE").collect()
    print("[OK] Inference service dropped.")
except Exception as e:
    print(f"[SKIP] Inference service: {e}")

# 2. Suspend the compute pool (stops billing but keeps config)
try:
    session.sql("ALTER COMPUTE POOL RECOMMEND_CPU_POOL SUSPEND").collect()
    print("[OK] Compute pool RECOMMEND_CPU_POOL suspended.")
except Exception as e:
    print(f"[SKIP] Compute pool suspend: {e}")

# 3. Drop the Feature Store online service (SPCS-backed Postgres)
try:
    fs.delete_online_service()
    print("[OK] Feature Store online service deleted.")
except Exception as e:
    print(f"[SKIP] Feature Store online service: {e}")

# 4. Suspend the warehouse
try:
    session.sql("ALTER WAREHOUSE RECOMMEND_WH SUSPEND").collect()
    print("[OK] Warehouse RECOMMEND_WH suspended.")
except Exception as e:
    print(f"[SKIP] Warehouse suspend: {e}")

print()
print("=== Done ===")
print("All running services and compute have been stopped.")
print("To fully remove all objects, run the cleanup section in setup.sql.")